In [ ]:
pip install pandas matplotlib

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("funnel_events.csv")

# Convert timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Funnel order
funnel_steps = [
    "visited_site",
    "signup_started",
    "details_filled",
    "email_verified",
    "purchase_completed"
]

# Remove duplicate events (same user reaching same step multiple times)
df = df.drop_duplicates(subset=["user_id", "step"])

# Count unique users at each step
counts = (
    df.groupby("step")["user_id"]
      .nunique()
      .reindex(funnel_steps, fill_value=0)
)

# Create funnel table
funnel = pd.DataFrame({
    "Step": counts.index,
    "Users": counts.values
})

# Conversion Rate
conversion = [100]

for i in range(1, len(funnel)):
    prev = funnel.loc[i-1, "Users"]
    curr = funnel.loc[i, "Users"]

    if prev == 0:
        conversion.append(0)
    else:
        conversion.append(round((curr / prev) * 100, 2))

funnel["Conversion Rate (%)"] = conversion

# Drop-off
dropoff = [0]

for i in range(1, len(funnel)):
    prev = funnel.loc[i-1, "Users"]
    curr = funnel.loc[i, "Users"]
    dropoff.append(prev - curr)

funnel["Drop-off Users"] = dropoff

print("\nFunnel Analysis")
print(funnel)

# Biggest drop-off
largest_drop = funnel.iloc[1:]["Drop-off Users"].idxmax()

print("\n===================================")
print("Biggest Drop-off Stage")
print(f"{funnel.loc[largest_drop-1,'Step']} ➜ {funnel.loc[largest_drop,'Step']}")
print(f"Users Lost: {funnel.loc[largest_drop,'Drop-off Users']}")
print("===================================")

# Visualization
plt.figure(figsize=(8,5))
plt.bar(funnel["Step"], funnel["Users"])
plt.title("Funnel Analysis")
plt.xlabel("Stage")
plt.ylabel("Unique Users")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# BONUS: Average time between stages
print("\nAverage Time Between Consecutive Steps")

for i in range(len(funnel_steps)-1):

    s1 = funnel_steps[i]
    s2 = funnel_steps[i+1]

    first = df[df["step"] == s1][["user_id","timestamp"]].rename(columns={"timestamp":"t1"})
    second = df[df["step"] == s2][["user_id","timestamp"]].rename(columns={"timestamp":"t2"})

    merged = first.merge(second,on="user_id")

    if len(merged):
        avg = (merged["t2"]-merged["t1"]).mean()
        print(f"{s1} -> {s2}: {avg}")

ModuleNotFoundError: No module named 'matplotlib'